# Complaint vs. Air Quality Correlation Analysis

**Goal**: Explore whether 311 air-pollution complaints in Chicago are related to
measured PM2.5 and NO2 levels from Open Air Chicago sensors.

This notebook is designed for **novice analysts** — each section explains what
we're doing and why, with no advanced statistics prerequisites.

---

### What you'll find
1. Load the merged dataset
2. Quick overview of the data
3. Average PM2.5 and NO2 by neighborhood
4. Complaint counts by neighborhood
5. Scatter plot: do neighborhoods with worse air get more complaints?
6. Correlation coefficients (how strong is the relationship?)
7. Time-lagged analysis: do complaints come *after* bad air days?
8. Summary and next steps

## Setup
Run this cell first to import all the libraries we need.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

# Nicer plots
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)

# Paths
import sys
sys.path.insert(0, str(Path("..").resolve()))
from scripts._common import MERGED_FILE, COMPLAINTS_CLEANED, NEIGHBORHOOD_SUMMARY
import aqi
print("Setup complete!")

## 1. Load the Merged Dataset

This file has one row per **sensor per day**.  Each row tells us:
- How high PM2.5 and NO2 were at that sensor that day
- How many 311 complaints were filed near that sensor that day
- Which neighborhood the sensor is in

In [ ]:
merged = pd.read_csv(MERGED_FILE, parse_dates=["date"])

print(f"Rows: {len(merged):,}")
print(f"Columns: {len(merged.columns)}")
print(f"Date range: {merged['date'].min().date()} to {merged['date'].max().date()}")
print(f"Sensors: {merged['sensor_name'].nunique()}")
print(f"Neighborhoods: {merged['neighborhood'].nunique()}")
print()
merged.head()

## 2. Quick Data Overview

Let's look at the basic statistics.  Key things to notice:
- **pm25_mean**: the daily average PM2.5 reading in µg/m³
- **no2_mean**: the daily average NO2 reading in ppb
- **complaint_count**: how many complaints were filed near this sensor that day (most days = 0)

In [ ]:
merged[["pm25_mean", "no2_mean", "complaint_count"]].describe().round(2)

In [ ]:
# How many days actually had complaints?
days_with_complaints = (merged["complaint_count"] > 0).sum()
total_days = len(merged)
print(f"Sensor-days with complaints: {days_with_complaints:,} / {total_days:,}  ({100*days_with_complaints/total_days:.1f}%)")
print(f"Total complaints: {merged['complaint_count'].sum():,}")

## 3. Average PM2.5 and NO2 by Neighborhood

Which neighborhoods have the worst air quality on average?

In [ ]:
# Neighborhood-level averages
neigh_air = (
    merged.groupby("neighborhood", as_index=False)
    .agg(
        avg_pm25=("pm25_mean", "mean"),
        avg_no2=("no2_mean", "mean"),
        total_complaints=("complaint_count", "sum"),
        sensor_count=("sensor_name", "nunique"),
        days_observed=("date", "nunique"),
    )
    .round(2)
    .sort_values("avg_pm25", ascending=False)
)

print(f"Neighborhoods with sensor data: {len(neigh_air)}")
neigh_air.head(15)

In [ ]:
# Top 20 neighborhoods by PM2.5
top20 = neigh_air.head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# PM2.5 bar chart
axes[0].barh(top20["neighborhood"], top20["avg_pm25"], color="#c44e52")
axes[0].set_xlabel("Average PM2.5 (µg/m³)")
axes[0].set_title("Top 20 Neighborhoods — Avg PM2.5")
axes[0].invert_yaxis()
axes[0].axvline(x=aqi.EPA_ANNUAL_STANDARD, color="orange", linestyle="--", label="EPA annual std (12)")
axes[0].legend()

# NO2 bar chart
axes[1].barh(top20["neighborhood"], top20["avg_no2"], color="#4c72b0")
axes[1].set_xlabel("Average NO2 (ppb)")
axes[1].set_title("Top 20 Neighborhoods — Avg NO2")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Complaint Counts by Neighborhood

Which neighborhoods file the most 311 air-pollution complaints?

In [ ]:
# Load the full complaint file for richer detail
complaints = pd.read_csv(COMPLAINTS_CLEANED, parse_dates=["complaint_date"])

comp_by_neigh = (
    complaints.groupby("neighborhood", as_index=False)
    .agg(total_complaints=("complaint_id", "nunique"))
    .sort_values("total_complaints", ascending=False)
)

# Bar chart
fig, ax = plt.subplots(figsize=(14, 8))
top_comp = comp_by_neigh.head(20)
ax.barh(top_comp["neighborhood"], top_comp["total_complaints"], color="#55a868")
ax.set_xlabel("Total 311 Air Pollution Complaints")
ax.set_title("Top 20 Neighborhoods by Complaint Count")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTotal complaints: {comp_by_neigh['total_complaints'].sum():,}")
print(f"Neighborhoods with complaints: {len(comp_by_neigh)}")

## 5. Scatter Plot — Air Quality vs. Complaints

The key question: **Do neighborhoods with higher pollution also have more complaints?**

Each dot below is one neighborhood. If complaints are related to air quality,
we'd expect to see dots going from bottom-left to top-right.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# PM2.5 vs Complaints
axes[0].scatter(neigh_air["avg_pm25"], neigh_air["total_complaints"],
                s=80, alpha=0.7, c="#c44e52", edgecolors="white")
axes[0].set_xlabel("Average PM2.5 (µg/m³)")
axes[0].set_ylabel("Total 311 Complaints")
axes[0].set_title("PM2.5 vs. Complaints by Neighborhood")

# Add trend line
mask = neigh_air["avg_pm25"].notna() & neigh_air["total_complaints"].notna()
if mask.sum() > 2:
    z = np.polyfit(neigh_air.loc[mask, "avg_pm25"], neigh_air.loc[mask, "total_complaints"], 1)
    p = np.poly1d(z)
    x_line = np.linspace(neigh_air["avg_pm25"].min(), neigh_air["avg_pm25"].max(), 50)
    axes[0].plot(x_line, p(x_line), "--", color="gray", label=f"Trend line")
    axes[0].legend()

# NO2 vs Complaints
axes[1].scatter(neigh_air["avg_no2"], neigh_air["total_complaints"],
                s=80, alpha=0.7, c="#4c72b0", edgecolors="white")
axes[1].set_xlabel("Average NO2 (ppb)")
axes[1].set_ylabel("Total 311 Complaints")
axes[1].set_title("NO2 vs. Complaints by Neighborhood")

mask2 = neigh_air["avg_no2"].notna() & neigh_air["total_complaints"].notna()
if mask2.sum() > 2:
    z2 = np.polyfit(neigh_air.loc[mask2, "avg_no2"], neigh_air.loc[mask2, "total_complaints"], 1)
    p2 = np.poly1d(z2)
    x_line2 = np.linspace(neigh_air["avg_no2"].min(), neigh_air["avg_no2"].max(), 50)
    axes[1].plot(x_line2, p2(x_line2), "--", color="gray", label="Trend line")
    axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Correlation Coefficients

A **correlation coefficient** measures how strongly two variables move together:
- **+1.0** = perfect positive relationship (one goes up, the other always goes up)
- **0.0** = no relationship at all
- **-1.0** = perfect negative relationship

We calculate two types:
- **Pearson**: assumes a straight-line relationship
- **Spearman**: works even if the relationship is curved (ranked correlation)

The **p-value** tells you whether the result could be due to random chance.
A p-value < 0.05 is conventionally considered "statistically significant".

In [ ]:
# --- Neighborhood-level correlation ---
print("=" * 60)
print("NEIGHBORHOOD-LEVEL CORRELATION")
print("(Each data point = one neighborhood's averages)")
print("=" * 60)

for pollutant, col in [("PM2.5", "avg_pm25"), ("NO2", "avg_no2")]:
    valid = neigh_air[[col, "total_complaints"]].dropna()
    if len(valid) < 3:
        print(f"\n{pollutant}: Not enough data points")
        continue
    
    pearson_r, pearson_p = stats.pearsonr(valid[col], valid["total_complaints"])
    spearman_r, spearman_p = stats.spearmanr(valid[col], valid["total_complaints"])
    
    print(f"\n{pollutant} vs. Complaints  (n = {len(valid)} neighborhoods)")
    print(f"  Pearson  r = {pearson_r:+.3f}   p = {pearson_p:.4f}  {'*' if pearson_p < 0.05 else ''}")
    print(f"  Spearman r = {spearman_r:+.3f}   p = {spearman_p:.4f}  {'*' if spearman_p < 0.05 else ''}")

In [ ]:
# --- Daily sensor-level correlation ---
print("=" * 60)
print("DAILY SENSOR-LEVEL CORRELATION")
print("(Each data point = one sensor on one day)")
print("=" * 60)

for pollutant, col in [("PM2.5", "pm25_mean"), ("NO2", "no2_mean")]:
    valid = merged[[col, "complaint_count"]].dropna()
    if len(valid) < 3:
        print(f"\n{pollutant}: Not enough data points")
        continue
    
    # Spearman is better here because complaint_count is very skewed (mostly 0)
    spearman_r, spearman_p = stats.spearmanr(valid[col], valid["complaint_count"])
    
    print(f"\n{pollutant} vs. Daily Complaints  (n = {len(valid):,} sensor-days)")
    print(f"  Spearman r = {spearman_r:+.3f}   p = {spearman_p:.2e}  {'*' if spearman_p < 0.05 else ''}")

print("\n(Note: daily correlation is often weaker because most days have 0 complaints)")

## 7. Time-Lagged Analysis

People might not complain on the exact day the air is bad — they might complain
the **next day** or even 2 days later.

This section checks:
- Do complaints go up the day **after** a PM2.5 spike?
- How do average complaint counts compare on spike days vs. normal days?

In [ ]:
# Compare complaint counts on spike vs. non-spike days
# A "spike" is defined as PM2.5 > 35 µg/m³ (EPA 24-hour standard)

spike = merged[merged["pm25_mean"] > aqi.PM25_SPIKE_THRESHOLD]
no_spike = merged[merged["pm25_mean"] <= aqi.PM25_SPIKE_THRESHOLD]

print(f"PM2.5 Spike days (> 35 µg/m³): {len(spike):,} sensor-days")
print(f"Non-spike days:                 {len(no_spike):,} sensor-days")
print()
print(f"Average complaints on SPIKE days:     {spike['complaint_count'].mean():.4f}")
print(f"Average complaints on NON-SPIKE days: {no_spike['complaint_count'].mean():.4f}")

if len(spike) > 0 and len(no_spike) > 0:
    ratio = spike['complaint_count'].mean() / max(no_spike['complaint_count'].mean(), 0.0001)
    print(f"\nRatio: {ratio:.2f}x more complaints on spike days")

In [ ]:
# Lag analysis: do complaints increase the day AFTER high PM2.5?
# We use the pre-computed lag columns from the pipeline

lag_data = merged.dropna(subset=["pm25_lag1", "complaint_count"]).copy()

# Create PM2.5 bins for yesterday's air quality
lag_data["pm25_yesterday_bin"] = pd.cut(
    lag_data["pm25_lag1"],
    bins=[0] + list(aqi.PM25_AQI_BREAKPOINTS) + [200],
    labels=[c[0] for c in aqi.AQI_CATEGORIES.values()][:5]
)

lag_summary = (
    lag_data.groupby("pm25_yesterday_bin", observed=True)
    .agg(
        avg_complaints_today=("complaint_count", "mean"),
        days_in_bin=("complaint_count", "count"),
        total_complaints=("complaint_count", "sum"),
    )
    .round(4)
)

print("Average complaints TODAY grouped by YESTERDAY's PM2.5 level:\n")
print(lag_summary.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
lag_summary["avg_complaints_today"].plot(kind="bar", ax=ax, color="#c44e52", edgecolor="white")
ax.set_xlabel("Yesterday's PM2.5 Level")
ax.set_ylabel("Avg Complaints Today")
ax.set_title("Do Complaints Follow Bad Air Days? (1-Day Lag)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Heatmap — Complaint Timing by Day of Week

Are complaints more common on certain days of the week?
This might reveal whether complaints are influenced by work schedules
or whether they truly follow air quality patterns.

In [ ]:
complaints["dow"] = complaints["complaint_date"].dt.day_name()
complaints["month_name"] = complaints["complaint_date"].dt.strftime("%b %Y")

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

pivot = (
    complaints.groupby(["month_name", "dow"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=dow_order)
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd", ax=ax)
ax.set_title("311 Air Complaints by Month and Day of Week")
ax.set_ylabel("Month")
ax.set_xlabel("Day of Week")
plt.tight_layout()
plt.show()

# Day-of-week totals
print("\nTotal complaints by day of week:")
print(complaints["dow"].value_counts().reindex(dow_order).to_string())

## 9. Neighborhood Comparison: Top Complaint Areas

For the neighborhoods with the most complaints, let's compare their
average air quality side by side.

In [ ]:
# Load the neighborhood summary (one row per neighborhood)
neigh_summary = pd.read_csv(NEIGHBORHOOD_SUMMARY)

# Top 15 neighborhoods by complaints
top_complaint_neighs = (
    neigh_summary
    .sort_values("total_complaints", ascending=False)
    .head(15)
    [["neighborhood", "total_complaints", "pm25_mean", "no2_mean", "sensor_count", "spike_days"]]
)

print("Top 15 neighborhoods by complaint count:\n")
print(top_complaint_neighs.to_string(index=False))

In [ ]:
# Side-by-side comparison chart
top15 = top_complaint_neighs.copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

axes[0].barh(top15["neighborhood"], top15["total_complaints"], color="#55a868")
axes[0].set_xlabel("Total Complaints")
axes[0].set_title("311 Complaints")
axes[0].invert_yaxis()

axes[1].barh(top15["neighborhood"], top15["pm25_mean"], color="#c44e52")
axes[1].set_xlabel("Avg PM2.5 (µg/m³)")
axes[1].set_title("Average PM2.5")
axes[1].invert_yaxis()
axes[1].axvline(x=aqi.EPA_ANNUAL_STANDARD, color="orange", linestyle="--", label="EPA std")
axes[1].legend()

axes[2].barh(top15["neighborhood"], top15["no2_mean"], color="#4c72b0")
axes[2].set_xlabel("Avg NO2 (ppb)")
axes[2].set_title("Average NO2")
axes[2].invert_yaxis()

fig.suptitle("Top 15 Complaint Neighborhoods — Air Quality Comparison", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 10. Summary & Interpretation Guide

### How to read the results

| What to look at | What it tells you |
|-----------------|-------------------|
| Scatter plots (Section 5) | If dots trend upward: complaints correlate with pollution |
| Correlation coefficients (Section 6) | Closer to ±1 = stronger relationship; p < 0.05 = statistically significant |
| Spike vs. non-spike (Section 7) | Higher ratio = complaints respond to bad air events |
| Lag analysis (Section 7) | Increasing bars left-to-right = people complain after bad air days |
| Day-of-week heatmap (Section 8) | Weekday clustering suggests complaints follow work schedules |
| Neighborhood comparison (Section 9) | Mismatches between complaints and PM2.5 may reveal other pollution sources |

### Common findings

- **Weak daily correlation** is normal — most sensors have 0 complaints on any given day
- **Neighborhood-level correlation** is usually more meaningful
- **Lag effects** matter — people don't complain instantly
- **Some high-complaint areas may have low PM2.5** — this could indicate odor or
  industrial complaints not captured by particle sensors

### Next steps

1. Filter to specific neighborhoods to zoom in on patterns
2. Try grouping by month to see seasonal effects
3. Look at complaint *text* (`complaint_detail`) to understand what people complain about
4. Overlay with Tableau dashboards for interactive exploration